## Problem Statement

### Context

Bike-sharing systems represent an evolution of traditional bike rental services where membership, rental, and return processes are fully automated. Users can pick up a bicycle from one docking station and return it to any other station within the network. Globally, more than 500 bike-sharing programs operate with over half a million bicycles. These systems play a significant role in reducing traffic congestion, lowering environmental impact, and promoting healthier lifestyles.

### Objective

'Travel Along' is an emerging bike-sharing company aiming to increase its customer base while delivering high-quality service at competitive prices. The company has gathered two years of operational data covering weather conditions, weekends, holidays, and hourly rental patterns.

As the newly appointed data scientist, your mandate is to:
- Uncover key patterns and factors that influence bike rental demand.
- Provide actionable business recommendations that help the company manage capacity during peak hours and grow sustainably.
- Build robust predictive models using ensemble techniques (Bagging, Boosting, and Stacking) to forecast the total number of bikes rented so that the company can plan inventory and staffing in advance.

### Data Description

| Feature | Description |
|---------|-------------|
| instant | Record index |
| dteday | Date |
| season | Season (1: Spring, 2: Summer, 3: Fall, 4: Winter) |
| yr | Year (0: 2011, 1: 2012) |
| mnth | Month (1 to 12) |
| hr | Hour of the day (0 to 23) |
| holiday | Whether the day is a holiday (1) or not (0) |
| weekday | Day of the week (0 = Sunday ... 6 = Saturday) |
| workingday | 1 if the day is neither weekend nor holiday, else 0 |
| weathersit | Weather situation: 1 Clear, 2 Mist, 3 Light Rain/Snow, 4 Heavy Rain/Snow |
| temp | Normalized temperature in Celsius |
| atemp | Normalized feeling temperature in Celsius |
| hum | Normalized humidity |
| windspeed | Normalized wind speed |
| casual | Count of casual users |
| registered | Count of registered users |
| cnt | Total rental bikes (casual + registered) – **Target Variable** |

## Importing Necessary Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    BaggingRegressor,
    RandomForestRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor,
    StackingRegressor
)
from xgboost import XGBRegressor
from sklearn import metrics

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: "%.3f" % x)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## Loading the Dataset

In [ ]:
data = pd.read_csv("hour.csv")
print("Shape of the dataset:", data.shape)
data.head()

## Data Preprocessing

In [ ]:
# Create a working copy
df = data.copy()

# Check missing values
print("Missing values:")
print(df.isnull().sum())

# Check duplicates
print("\nNumber of duplicate rows:", df.duplicated().sum())

# Drop unnecessary columns
df.drop(columns=["instant", "dteday"], inplace=True)

print("\nShape after dropping columns:", df.shape)
print("Columns:", df.columns.tolist())

## Exploratory Data Analysis (EDA)

### Univariate Analysis – Target Variable (cnt)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["cnt"], kde=True, ax=ax[0], color="steelblue")
ax[0].set_title("Distribution of Total Bike Rentals (cnt)")

sns.boxplot(x=df["cnt"], ax=ax[1], color="lightcoral")
ax[1].set_title("Boxplot of Total Bike Rentals")

plt.tight_layout()
plt.show()

print(f"Mean: {df['cnt'].mean():.2f}, Median: {df['cnt'].median():.2f}")
print(f"Min: {df['cnt'].min()}, Max: {df['cnt'].max()}")

**Observations:**
- Distribution is right-skewed.
- Most hours have relatively low rental counts.
- Maximum hourly rentals reach 977 bikes.

### Univariate Analysis – Categorical Features

In [ ]:
cat_cols = ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit"]

fig, axes = plt.subplots(4, 2, figsize=(16, 18))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.countplot(x=col, data=df, ax=axes[i], palette="viridis")
    axes[i].set_title(f"Count of {col}")
    axes[i].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

**Observations:**
- Season distribution is fairly balanced.
- Holidays are rare.
- Working days dominate.
- Clear weather is most frequent; severe weather is rare.

### Bivariate Analysis – Rentals by Season, Year, Month, Hour

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.boxplot(x="season", y="cnt", data=df, ax=axes[0, 0], palette="Set2")
axes[0, 0].set_title("Bike Rentals by Season")

sns.boxplot(x="yr", y="cnt", data=df, ax=axes[0, 1], palette="Set1")
axes[0, 1].set_title("Bike Rentals by Year")

sns.boxplot(x="mnth", y="cnt", data=df, ax=axes[1, 0], palette="coolwarm")
axes[1, 0].set_title("Bike Rentals by Month")

sns.boxplot(x="hr", y="cnt", data=df, ax=axes[1, 1], palette="magma")
axes[1, 1].set_title("Bike Rentals by Hour of Day")

plt.tight_layout()
plt.show()

**Observations:**
- Fall and Summer have highest rentals; Spring has lowest.
- 2012 has higher rentals than 2011.
- Clear morning (~8 AM) and evening (~5-6 PM) peaks.

### Rentals by Holiday, Working Day and Weather

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(x="holiday", y="cnt", data=df, ax=axes[0], palette="pastel")
axes[0].set_title("Holiday vs Non-Holiday")

sns.boxplot(x="workingday", y="cnt", data=df, ax=axes[1], palette="pastel")
axes[1].set_title("Working Day vs Non-Working Day")

sns.boxplot(x="weathersit", y="cnt", data=df, ax=axes[2], palette="pastel")
axes[2].set_title("Rentals by Weather")

plt.tight_layout()
plt.show()

**Observations:**
- Holidays have lower rentals.
- Working days have higher demand.
- Clear weather has highest rentals; demand drops with worse weather.

### Continuous Features vs Target

In [ ]:
num_cols = ["temp", "atemp", "hum", "windspeed"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.scatterplot(x=col, y="cnt", data=df, ax=axes[i], alpha=0.3, color="teal")
    axes[i].set_title(f"cnt vs {col}")

plt.tight_layout()
plt.show()

**Observations:**
- Temperature has positive relationship with rentals.
- Humidity and windspeed show mild negative trends.

### Correlation Heatmap

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="Spectral", center=0, linewidths=0.5)
plt.title("Correlation Matrix")
plt.show()

**Observations:**
- `temp` and `atemp` are almost perfectly correlated → drop one.
- `casual` + `registered` = `cnt` → drop both before modelling.
- `hr`, `temp`, `yr` have strongest positive correlations with `cnt`.

## EDA Summary

- Demand is highly time-dependent (morning and evening peaks).
- Fall and Summer have highest rentals; Spring has lowest.
- 2012 shows higher demand than 2011.
- Clear weather maximises demand.
- Drop `casual`, `registered` (leakage) and `atemp` (multicollinearity).

## Feature Engineering

In [ ]:
# Drop leakage and redundant columns
df.drop(columns=["casual", "registered", "atemp"], inplace=True)

# Peak hour flag (morning 7-9 and evening 17-19)
df["peak_hour"] = df["hr"].apply(lambda x: 1 if (7 <= x <= 9) or (17 <= x <= 19) else 0)

# Bad weather flag
df["bad_weather"] = df["weathersit"].apply(lambda x: 1 if x >= 3 else 0)

print("Columns after feature engineering:")
print(df.columns.tolist())
print("Shape:", df.shape)

## Data Preparation for Modelling

In [ ]:
X = df.drop(columns=["cnt"])
y = df["cnt"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

print("Training set:", X_train.shape[0])
print("Test set    :", X_test.shape[0])

## Model Building – Ensemble Techniques

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name="Model"):
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test  = model.predict(X_test)
    
    metrics_dict = {
        "Model": model_name,
        "Train RMSE": np.sqrt(metrics.mean_squared_error(y_train, y_pred_train)),
        "Test RMSE" : np.sqrt(metrics.mean_squared_error(y_test, y_pred_test)),
        "Train MAE" : metrics.mean_absolute_error(y_train, y_pred_train),
        "Test MAE"  : metrics.mean_absolute_error(y_test, y_pred_test),
        "Train R2"  : metrics.r2_score(y_train, y_pred_train),
        "Test R2"   : metrics.r2_score(y_test, y_pred_test)
    }
    return metrics_dict, model

### 1. Decision Tree (Baseline)

In [ ]:
dt_model = DecisionTreeRegressor(random_state=42)
dt_metrics, dt_fitted = evaluate_model(dt_model, X_train, y_train, X_test, y_test, "Decision Tree")
pd.DataFrame([dt_metrics])

### 2. Bagging

In [ ]:
bag_model = BaggingRegressor(
    estimator=DecisionTreeRegressor(random_state=42),
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)
bag_metrics, bag_fitted = evaluate_model(bag_model, X_train, y_train, X_test, y_test, "Bagging")
pd.DataFrame([bag_metrics])

### 3. Random Forest – Hyperparameter Tuning

In [ ]:
rf_params = {
    "n_estimators": [100, 150],
    "max_depth": [15, 20],
    "min_samples_split": [2, 5]
}

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=rf_params,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)
print("Best parameters:", rf_grid.best_params_)

rf_metrics, rf_fitted = evaluate_model(
    rf_grid.best_estimator_, X_train, y_train, X_test, y_test, "Random Forest (Tuned)"
)
pd.DataFrame([rf_metrics])

### 4. AdaBoost

In [ ]:
ada_model = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(max_depth=8, random_state=42),
    n_estimators=50,
    learning_rate=0.1,
    random_state=42
)
ada_metrics, ada_fitted = evaluate_model(ada_model, X_train, y_train, X_test, y_test, "AdaBoost")
pd.DataFrame([ada_metrics])

### 5. Gradient Boosting – Hyperparameter Tuning

In [ ]:
gb_params = {
    "n_estimators": [100, 150],
    "learning_rate": [0.05, 0.1],
    "max_depth": [4, 5]
}

gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid=gb_params,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

gb_grid.fit(X_train, y_train)
print("Best parameters:", gb_grid.best_params_)

gb_metrics, gb_fitted = evaluate_model(
    gb_grid.best_estimator_, X_train, y_train, X_test, y_test, "Gradient Boosting (Tuned)"
)
pd.DataFrame([gb_metrics])

### 6. XGBoost – Hyperparameter Tuning

In [ ]:
xgb_params = {
    "n_estimators": [100, 150],
    "learning_rate": [0.05, 0.1],
    "max_depth": [5, 6]
}

xgb_grid = GridSearchCV(
    XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
    param_grid=xgb_params,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

xgb_grid.fit(X_train, y_train)
print("Best parameters:", xgb_grid.best_params_)

xgb_metrics, xgb_fitted = evaluate_model(
    xgb_grid.best_estimator_, X_train, y_train, X_test, y_test, "XGBoost (Tuned)"
)
pd.DataFrame([xgb_metrics])

### 7. Stacking

In [ ]:
estimators = [
    ("rf", RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)),
    ("gb", GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)),
    ("xgb", XGBRegressor(n_estimators=100, max_depth=5, random_state=42, verbosity=0))
]

stack_model = StackingRegressor(
    estimators=estimators,
    final_estimator=GradientBoostingRegressor(n_estimators=50, random_state=42),
    n_jobs=-1
)

stack_metrics, stack_fitted = evaluate_model(
    stack_model, X_train, y_train, X_test, y_test, "Stacking"
)
pd.DataFrame([stack_metrics])

## Model Comparison

In [ ]:
all_metrics = [dt_metrics, bag_metrics, rf_metrics, ada_metrics, gb_metrics, xgb_metrics, stack_metrics]
comparison_df = pd.DataFrame(all_metrics).sort_values(by="Test R2", ascending=False).reset_index(drop=True)
comparison_df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x="Test R2", y="Model", data=comparison_df, ax=ax[0], palette="viridis")
ax[0].set_title("Test R² by Model")
ax[0].set_xlim(0, 1)

sns.barplot(x="Test RMSE", y="Model", data=comparison_df, ax=ax[1], palette="magma")
ax[1].set_title("Test RMSE by Model")

plt.tight_layout()
plt.show()

## Feature Importance

In [ ]:
importances = pd.Series(rf_fitted.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=importances.values, y=importances.index, palette="coolwarm")
plt.title("Feature Importance – Random Forest")
plt.tight_layout()
plt.show()

print(importances)

## Business Recommendations

1. **Dynamic Fleet Allocation** – Concentrate bikes during morning (7–9 AM) and evening (5–7 PM) using the peak_hour feature and model predictions.

2. **Seasonal Capacity Planning** – Increase bikes and staff in Fall and Summer; reduce fleet or schedule maintenance in Spring.

3. **Weather-Responsive Operations** – Prepare for lower demand on rainy/snowy days (bad_weather flag).

4. **Targeted Marketing** – Promote membership to convert casual weekend riders.

5. **Model Deployment** – Deploy the best-performing tuned model for hourly demand forecasting.

## Conclusion

This case study covered data preprocessing, EDA, simple feature engineering, ensemble modelling with hyperparameter tuning, and business recommendations.

Key takeaways:
- Hour of day, season, year and weather are the strongest drivers of demand.
- Simple engineered features (peak_hour, bad_weather) improve the model.
- Tuned ensemble models deliver high predictive accuracy (Test R² typically > 0.90).
- Removing leakage and collinear features is essential for a realistic model.

Deploying the chosen model will help Travel Along optimise bike distribution and support sustainable growth.